# Spatially explicit models: geography in the model

```{admonition} Question
:class: tip
Which of our models is spatially explicit?
```

**In this notebook:** 1 · the spatial lag  2 · the neighbours alone  3 · three models  4 · moved places  5 · the invariance test  6 · smaller hexagons

## 1 · The spatial lag

A table has no geography. The **spatial lag** of week 3 writes the map into it: a column's average over the hexagons that touch a hexagon.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

import helpers as h

hexes, X, y, W = h.load()        # the table, the features, the labels, the neighbours
SEED = 0
X_around = W @ X                 # the spatial lag of all 16 features: the neighbours' profile
xy = hexes[["x", "y"]].to_numpy()                # the centre of every hexagon, in metres

Three counts of the five hexagons, each with its spatial lag:

In [ ]:
five = h.five(hexes)             # place name -> row number
lag_rows = pd.DataFrame({"place": five.index}, index=range(1, 6))
for f in ["car", "play", "office"]:
    lag_rows[f] = hexes[f].to_numpy()[five.values]
    lag_rows[f + ", neighbours' average"] = (W @ hexes[f].to_numpy(float))[five.values].round(1)
lag_rows["landuse"] = y[five.values]
lag_rows

## 2 · The neighbours alone

The boundary rule of the supervised page (logistic regression, all 16 counts) gets three descriptions of a hexagon. Each is scored on the test pile, as the mean of 20 random deals.

In [ ]:
def test_accuracy(feats, deals=20):
    """Fit the boundary rule on the training pile and score it on the test pile: the mean of 20 random deals."""
    scores = []
    for seed in range(deals):
        tr, va, te = h.three_piles(y, seed)
        scores.append(LogisticRegression(max_iter=3000).fit(feats[tr], y[tr]).score(feats[te], y[te]))
    return float(np.mean(scores))


described = {"own profile": X, "own profile + neighbours' profile": np.hstack([X, X_around]), "neighbours' profile only": X_around}
lag_scores = {name: test_accuracy(feats) for name, feats in described.items()}
for name, score in lag_scores.items():
    print(f"{name:36s} {score:.1%} on the test pile (mean of 20 deals)")
print(f"always answering 'green'             {(y == 'green').mean():.1%}")

**Result.** The neighbours alone, without one fact about the hexagon itself, reach 75%. The hexagon's own counts reach 79%, and both together 80%: the lag adds about one point.

## 3 · Three models

A model is **spatially explicit** when location enters it. Two simple ways: add the neighbours' profile, or add the coordinates of the hexagon centre. The model is a random forest: 200 decision trees that vote.

In [ ]:
versions = {
    "POIs only": X,
    "POIs + neighbours' profile": np.hstack([X, X_around]),
    "POIs + coordinates": np.hstack([X, xy]),
}
train, test = train_test_split(np.arange(len(y)), test_size=0.3, stratify=y, random_state=SEED)
models = {name: h.new_forest().fit(F[train], y[train]) for name, F in versions.items()}
before = {name: models[name].predict(F[test]) for name, F in versions.items()}
for name in versions:
    print(f"{name:28s} accuracy {(before[name] == y[test]).mean():.1%} on the test hexagons")

## 4 · Moved places

Goodchild's **invariance test**: move the places and see whether the results change. Every hexagon keeps its counts and its label and goes to a random position. Its neighbours and its coordinates are new.

In [ ]:
rng = np.random.default_rng(SEED)
new_position = rng.permutation(len(y))        # hexagon i moves to where hexagon new_position[i] used to be
occupant = np.argsort(new_position)           # occupant[p] = the hexagon that now sits at position p
moved = {
    "POIs only": X,                                                               # nothing in here knows about location
    "POIs + neighbours' profile": np.hstack([X, (W @ X[occupant])[new_position]]),  # profile of the NEW neighbours
    "POIs + coordinates": np.hstack([X, xy[new_position]]),                       # the NEW coordinates
}
h.plot_relocation(hexes, y, occupant);

## 5 · The invariance test

Two questions. Does the trained model answer differently on the moved places? Does a model that is trained again on the moved places score lower than on the real map?

In [ ]:
after = {name: models[name].predict(F[test]) for name, F in moved.items()}                                  # question 1
retrained = {name: h.new_forest().fit(F[train], y[train]).predict(F[test]) for name, F in moved.items()}    # question 2
acc = lambda pred: (pred == y[test]).mean()
result = pd.DataFrame({"answers that change": {n: (before[n] != after[n]).mean() for n in versions},
                       "accuracy on the real map": {n: acc(before[n]) for n in versions},
                       "same model, moved places": {n: acc(after[n]) for n in versions},
                       "trained again on moved places": {n: acc(retrained[n]) for n in versions}})
h.plot_invariance(result)
result.map("{:.0%}".format)

**Result.** 0%, 14% and 3% of the answers change: the two models that were given location are spatially explicit. Neither is more accurate, on the real map or trained again on the moved places: 80% to 81%.

## 6 · Smaller hexagons

At H3 resolution 9 a hexagon is seven times smaller. What the real arrangement is worth to the model with the neighbours' profile: its accuracy on the real map, minus its accuracy when trained on moved places.

In [ ]:
def worth_of_the_arrangement(res, relocations=3):
    """The forest with the neighbours' profile: accuracy on the real map, and trained on moved places (mean of 3 moves)."""
    g, Xg, yg, Wg = h.load(res)
    tr, te = train_test_split(np.arange(len(yg)), test_size=0.3, stratify=yg, random_state=SEED)
    fit_score = lambda F: (h.new_forest(150).fit(F[tr], yg[tr]).predict(F[te]) == yg[te]).mean()
    real = fit_score(np.hstack([Xg, Wg @ Xg]))
    scrambled = []
    for s in range(relocations):
        pos = np.random.default_rng(s).permutation(len(yg))
        scrambled.append(fit_score(np.hstack([Xg, (Wg @ Xg[np.argsort(pos)])[pos]])))
    return {"hexagons": len(g), "median POIs per hexagon": g["poi_total"].median(), "POIs only": fit_score(Xg),
            "with neighbours, real map": real, "with neighbours, relocated map": np.mean(scrambled),
            "worth of the arrangement (points)": (real - np.mean(scrambled)) * 100}

In [ ]:
try:
    by_size = pd.DataFrame({f"resolution {r}": worth_of_the_arrangement(r) for r in (8, 9)}).T
    h.plot_unit_size(by_size)
    display(by_size.round(3))
except FileNotFoundError as err:      # the table of resolution 9 is not in the folder data/
    print(err)

**Result.** At resolution 8 the median hexagon holds 15 POIs and the arrangement is worth 0 points. At resolution 9 it holds 1 POI and the neighbours are worth 4.3 points. The answer depends on the unit (the modifiable areal unit problem, week 1). Features and labels were built again for the smaller hexagons, so more than the size differs.

**Next:** [challenges](challenges): is a random deal of the piles a fair test?